# Sesión 5 - Implementación de algoritmos DQN y PG

> En la presente sesión, se va a desarrollar la **implementación** de los algoritmos básicos de aprendizaje por refuerzo: **Deep Q-networks (DQN)**, y **Policy Gradients (PG)**. El caso de uso desarrollado consistirá en el **Atari**, cuyo **entorno** es modelado por la librería de acceso público de **OpenAI gym**. Se hará uso de dos librería para la implementación de algoritmos de aprendizaje por refuerzo: **keras-rl** y **pytorch**. La primera, de más alto nivel, servirá para establecer un conocimiento práctico de los métodos DQN, y analizar los distintos hiperparámetros de entrenamiento. En el segundo caso, PG será implemetado con pytorch, librería de más bajo nivel, y la cuál permitirá trabajar en detalle la estrategia de aprendizaje on-policy y la optimización por gradiente de la policy.






---
## **PARTE 1** - Instalación y requisitos previos

> Las prácticas han sido preparadas para poder realizarse en el entorno de trabajo de Google Colab. Sin embargo, esta plataforma presenta ciertas incompatibilidades a la hora de visualizar la renderización en gym. Por ello, para obtener estas visualizaciones, se deberá trasladar el entorno de trabajo a local. Por ello, el presente dosier presenta instrucciones para poder trabajar en ambos entornos. Siga los siguientes pasos para un correcto funcionamiento:
1.   **LOCAL:** Preparar el enviroment, siguiendo las intrucciones detalladas en la sección *1.1.Preparar enviroment*.
2.  **AMBOS:** Modificar las variables "mount" y "drive_mount" a la carpeta de trabajo en drive en el caso de estar en Colab, y ejecturar la celda *1.2.Localizar entorno de trabajo*.
3. **COLAB:** se deberá ejecutar las celdas correspondientes al montaje de la carpeta de trabajo en Drive. Esta corresponde a la sección *1.3.Montar carpeta de datos local*.
4.  **AMBOS:** Instalar las librerías necesarias, siguiendo la sección *1.4.Instalar librerías necesarias*.



---
### 1.1. Preparar enviroment (solo local)



> Para preparar el entorno de trabajo en local, se han seguido los siguientes pasos:
1. En Windows, puede ser necesario instalar las C++ Build Tools. Para ello, siga los siguientes pasos: https://towardsdatascience.com/how-to-install-openai-gym-in-a-windows-environment-338969e24d30.
2. Instalar Anaconda
3. Siguiendo el código que se presenta comentado en la próxima celda: Crear un enviroment, cambiar la ruta de trabajo, e instalar librerías básicas.


```
conda update --all
conda create --name miar_rl python=3.8
conda activate miar_rl
cd "PATH_TO_FOLDER"
conda install git
pip install jupyter
```


4. Abrir la notebook con *jupyter-notebook*.



```
jupyter-notebook
```




---
### 1.2. Localizar entorno de trabajo: Google colab o local

In [1]:
# ATENCIÓN!! Modificar ruta relativa a la práctica si es distinta (drive_root)
mount='/content/gdrive'
# drive_root = mount + "/My Drive/VIU/08_AR_MIAR/sesiones_practicas/sesion_practica_1"
drive_root = mount + "/My Drive/VIU_Apr_por_Refuerzo_Grupo_17_2025_2026"


try:
  from google.colab import drive
  IN_COLAB=True
except:
  IN_COLAB=False

---
### 1.3. Montar carpeta de datos local (solo Colab)

In [2]:
# Switch to the directory on the Google Drive that you want to use
import os
if IN_COLAB:
  print("We're running Colab")

  if IN_COLAB:
    # Mount the Google Drive at mount
    print("Colab: mounting Google drive on ", mount)

    drive.mount(mount)

    # Create drive_root if it doesn't exist
    create_drive_root = True
    if create_drive_root:
      print("\nColab: making sure ", drive_root, " exists.")
      os.makedirs(drive_root, exist_ok=True)

    # Change to the directory
    print("\nColab: Changing directory to ", drive_root)
    %cd $drive_root
# Verify we're in the correct working directory
%pwd
print("Archivos en el directorio: ")
print(os.listdir())

We're running Colab
Colab: mounting Google drive on  /content/gdrive
Mounted at /content/gdrive

Colab: making sure  /content/gdrive/My Drive/VIU_Apr_por_Refuerzo_Grupo_17_2025_2026  exists.

Colab: Changing directory to  /content/gdrive/My Drive/VIU_Apr_por_Refuerzo_Grupo_17_2025_2026
/content/gdrive/My Drive/VIU_Apr_por_Refuerzo_Grupo_17_2025_2026
Archivos en el directorio: 
['08MIAR_dqn.ipynb', 'Proyecto_práctico_Marina.ipynb', 'Proyecto_práctico_Angel.ipynb', 'Proyecto_práctico_Williams.ipynb', 'Proyecto_práctico_Final.ipynb', 'Proyecto_práctico_Maria.ipynb', 'LICENSE', 'README.md', '.git', '08MIAR_dqn_tf2_18.ipynb', '202504_08MIAR_Grupos_Trabajo.pdf', 'dqn_CartPole-v0_weights.h5f.data-00000-of-00001', 'dqn_CartPole-v0_weights.h5f.index', 'dqn_BreakoutDeterministic-v4_weights.h5f.data-00000-of-00001', 'dqn_BreakoutDeterministic-v4_weights.h5f.index', 'dqn_BreakoutDeterministic-v4_log.json', 'checkpoint']


---
### 1.4. Instalar librerías necesarias


In [3]:
#bigframes 2.10.0 requires cloudpickle>=2.0.0, but you have cloudpickle 1.6.0 which is incompatible.
#dask 2024.12.1 requires cloudpickle>=3.0.0, but you have cloudpickle 1.6.0 which is incompatible.
#distributed 2024.12.1 requires cloudpickle>=3.0.0,


if IN_COLAB:
  %pip install tensorflow==2.18.0
  %pip install tf-keras==2.18.0
  %pip install gym==0.17.3
  %pip install git+https://github.com/Kojoley/atari-py.git
  %pip install keras-rl2==1.0.5
else:
  %pip install typing_extensions==3.7.4
  %pip install gym==0.17.3
  %pip install git+https://github.com/Kojoley/atari-py.git
  %pip install pyglet==1.5.0
  %pip install h5py==3.1.0
  %pip install Pillow==9.5.0
  %pip install tensorflow==2.5.3
  %pip install torch==2.0.1
  %pip install keras-rl2==1.0.5
  %pip install Keras==2.2.4
  %pip install agents==1.4.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
  Created wheel for gym: filename=gym-0.17.3-py3-none-any.whl size=1654617 sha256=896b2e507afcc018b6b4668b84427d50b66a1cf44cc558e817946e4446dd18bd
  Stored in directory: /root/.cache/pip/wheels/07/8b/b7/570cb90b10f17e85ccb291ba1f04af41ec697745104a2263eb
Successfully built gym
  Attempting uninstall: cloudpickle
    Found existing installation: cloudpickle 3.1.1
    Uninstalling cloudpickle-3.1.1:
      Successfully uninstalled cloudpickle-3.1.1
  Attempting uninstall: gym
    Found existing installation: gym 0.25.2
    Uninstalling gym-0.25.2:
      Successfully uninstalled gym-0.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.10.0 requires cloudpickl

In [4]:
import os
os.environ['TF_USE_LEGACY_KERAS']="1"

In [5]:
#Comprobamos que se carga la versión esperada de TF y detecta la GPU
import tensorflow as tf1
print(tf1.__version__)
print(tf1.config.list_physical_devices('GPU'))
print(tf1.config.list_logical_devices('GPU'))

2.18.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[LogicalDevice(name='/device:GPU:0', device_type='GPU')]


In [6]:
import tensorflow.keras as tf
from keras import __version__
tf.__version__ = __version__

---
### 1.5.Acerca de las librerías para RL

Librería para trabajar con nuestros entornos: gym (https://gym.openai.com/) \
Librería para trabajar con deep learning: tensorflow (https://www.tensorflow.org/) \
Librería para desarrollar soluciones de RL a alto nivel: keras-rl (https://github.com/keras-rl/keras-rl) \


---
## **PARTE 2** - *Deep Q-learning*


---
### 2.1. DQN Pseudo-código

In [ ]:
from IPython import display
display.Image("images/dqn.png", width = 600, height = 300)

---
### 2.2. Ejemplo de DQN with keras-rl - CartPole

¿Quieres intentar estabilizar el péndulo invertido tu?
https://jeffjar.me/cartpole.html

Información del entorno:
https://www.gymlibrary.dev/environments/classic_control/cart_pole/

Basado en: https://github.com/keras-rl/keras-rl/blob/master/examples/dqn_cartpole.py

In [7]:
import numpy as np
import gym

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, InputLayer

if IN_COLAB:
  from tensorflow.keras.optimizers.legacy import Adam
else:
  from tensorflow.keras.optimizers import Adam

from rl.agents.dqn import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [8]:
ENV_NAME = 'CartPole-v0'

# Get the environment and extract the number of actions.
env = gym.make(ENV_NAME)
np.random.seed(123)
env.seed(123)
nb_actions = env.action_space.n

In [9]:
print("Numero de acciones disponibles:" + str(nb_actions))

Numero de acciones disponibles:2


In [10]:
print("Formato de las observaciones:")
env.observation_space

Formato de las observaciones:


Box(-3.4028234663852886e+38, 3.4028234663852886e+38, (4,), float32)

In [11]:
# Next, we build a very simple model.
model = Sequential()
model.add(Flatten(input_shape=(1,)+env.observation_space.shape))
model.add(Dense(16))
model.add(Activation('relu'))
model.add(Dense(16))
model.add(Activation('relu'))
model.add(Dense(16))
model.add(Activation('relu'))
model.add(Dense(nb_actions))
model.add(Activation('linear'))
print(model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 4)                 0         
                                                                 
 dense (Dense)               (None, 16)                80        
                                                                 
 activation (Activation)     (None, 16)                0         
                                                                 
 dense_1 (Dense)             (None, 16)                272       
                                                                 
 activation_1 (Activation)   (None, 16)                0         
                                                                 
 dense_2 (Dense)             (None, 16)                272       
                                                                 
 activation_2 (Activation)   (None, 16)                0

In [12]:
# Let's define the memory for storing the experience
memory = SequentialMemory(limit=50000, window_length=1)

In [13]:
# Define the policy that our agent will follow
policy = BoltzmannQPolicy()

In [14]:
# Define the agent
dqn = DQNAgent(model=model, nb_actions=nb_actions, memory=memory,
               nb_steps_warmup=10,
               target_model_update=1e-2, policy=policy)
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])

In [15]:
# Train the agent
dqn.fit(env, nb_steps=50000, visualize=False, verbose=2)

# After training is done, we save the final weights.
dqn.save_weights('dqn_{}_weights.h5f'.format(ENV_NAME), overwrite=True)

Training for 50000 steps ...


/usr/local/lib/python3.11/dist-packages/tf_keras/src/engine/training_v1.py:2354: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
/usr/local/lib/python3.11/dist-packages/rl/memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


    26/50000: episode: 1, duration: 3.001s, episode steps:  26, steps per second:   9, episode reward: 26.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.538 [0.000, 1.000],  loss: 0.465469, mae: 0.504150, mean_q: 0.022518


/usr/local/lib/python3.11/dist-packages/rl/memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


    64/50000: episode: 2, duration: 0.422s, episode steps:  38, steps per second:  90, episode reward: 38.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.632 [0.000, 1.000],  loss: 0.393055, mae: 0.500331, mean_q: 0.205993
    88/50000: episode: 3, duration: 0.257s, episode steps:  24, steps per second:  93, episode reward: 24.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.458 [0.000, 1.000],  loss: 0.293711, mae: 0.539804, mean_q: 0.494024
   117/50000: episode: 4, duration: 0.300s, episode steps:  29, steps per second:  97, episode reward: 29.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.517 [0.000, 1.000],  loss: 0.137321, mae: 0.589802, mean_q: 0.857655
   125/50000: episode: 5, duration: 0.086s, episode steps:   8, steps per second:  92, episode reward:  8.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.875 [0.000, 1.000],  loss: 0.053394, mae: 0.685036, mean_q: 1.217473
   148/50000: episode: 6, duration: 0.411s, episode steps:  23, step

/usr/local/lib/python3.11/dist-packages/rl/core.py:206: RuntimeWarning: overflow encountered in scalar add
  self.step += 1


 32962/50000: episode: 205, duration: 0.372s, episode steps: 200, steps per second: 538, episode reward: 200.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.500 [0.000, 1.000],  loss: 31.370708, mae: 44.155291, mean_q: 88.018725
 33162/50000: episode: 206, duration: 0.294s, episode steps: 200, steps per second: 680, episode reward: 200.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.490 [0.000, 1.000],  loss: --, mae: --, mean_q: --
 33362/50000: episode: 207, duration: 0.297s, episode steps: 200, steps per second: 672, episode reward: 200.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.500 [0.000, 1.000],  loss: --, mae: --, mean_q: --
 33562/50000: episode: 208, duration: 0.309s, episode steps: 200, steps per second: 647, episode reward: 200.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.495 [0.000, 1.000],  loss: --, mae: --, mean_q: --
 33762/50000: episode: 209, duration: 0.289s, episode steps: 200, steps per second: 692, episode reward: 20

In [16]:
# Finally, evaluate our algorithm for 5 episodes.
dqn.load_weights('dqn_{}_weights.h5f'.format(ENV_NAME))
dqn.test(env, nb_episodes=10, visualize=False)


Testing for 10 episodes ...
Episode 1: reward: 200.000, steps: 200
Episode 2: reward: 200.000, steps: 200
Episode 3: reward: 200.000, steps: 200
Episode 4: reward: 200.000, steps: 200
Episode 5: reward: 200.000, steps: 200
Episode 6: reward: 200.000, steps: 200
Episode 7: reward: 200.000, steps: 200
Episode 8: reward: 200.000, steps: 200
Episode 9: reward: 200.000, steps: 200
Episode 10: reward: 200.000, steps: 200


---
### 2.3. Ejemplo DQN con Keras-rl - Breakout

Información del entorno: https://www.gymlibrary.dev/environments/atari/breakout/

Basado en: https://github.com/Finspire13/pytorch-policy-gradient-example/blob/master/pg.py

In [17]:
from __future__ import division

from PIL import Image
import numpy as np
import gym

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, Convolution2D, Permute
from tensorflow.keras.optimizers.legacy import Adam
import tensorflow.keras.backend as K

from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import FileLogger, ModelIntervalCheckpoint

In [18]:
INPUT_SHAPE = (84, 84)
WINDOW_LENGTH = 4

In [19]:
# In this example, we need to preprocess the observations
class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3  # (height, width, channel)
        img = Image.fromarray(observation)
        img = img.resize(INPUT_SHAPE).convert('L')
        processed_observation = np.array(img)
        assert processed_observation.shape == INPUT_SHAPE
        return processed_observation.astype('uint8')

    def process_state_batch(self, batch):
        processed_batch = batch.astype('float32') / 255.
        return processed_batch

    def process_reward(self, reward):
        return np.clip(reward, -1., 1.)

In [20]:
# Get the environment and extract the number of actions.
import gym
import numpy as np
env_name = 'BreakoutDeterministic-v4'
env = gym.make(env_name)
np.random.seed(123)
env.seed(123)
nb_actions = env.action_space.n

In [21]:
print("Numero de acciones disponibles: " + str(nb_actions))

Numero de acciones disponibles: 4


In [22]:
print("Formato de las observaciones:")
env.observation_space

Formato de las observaciones:


Box(0, 255, (210, 160, 3), uint8)

In [23]:
# Next, we build our model. We use the same model that was described by Mnih et al. (2015).
input_shape = (WINDOW_LENGTH,) + INPUT_SHAPE
model = Sequential()
print(K.image_data_format())
if K.image_data_format() == 'channels_last':
    # (width, height, channels)
    model.add(Permute((2, 3, 1), input_shape=input_shape))
elif K.image_data_format() == 'channels_first':
    # (channels, width, height)
    model.add(Permute((1, 2, 3), input_shape=input_shape))
else:
    raise RuntimeError('Unknown image_dim_ordering.')

model.add(Convolution2D(32, (8, 8), strides=(4, 4)))
model.add(Activation('relu'))
model.add(Convolution2D(64, (4, 4), strides=(2, 2)))
model.add(Activation('relu'))
model.add(Convolution2D(64, (3, 3), strides=(1, 1)))
model.add(Activation('relu'))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dense(nb_actions))
model.add(Activation('linear'))
print(model.summary())

channels_last
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 permute (Permute)           (None, 84, 84, 4)         0         
                                                                 
 conv2d (Conv2D)             (None, 20, 20, 32)        8224      
                                                                 
 activation_4 (Activation)   (None, 20, 20, 32)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 9, 64)          32832     
                                                                 
 activation_5 (Activation)   (None, 9, 9, 64)          0         
                                                                 
 conv2d_2 (Conv2D)           (None, 7, 7, 64)          36928     
                                                                 
 activation_6 (Activation)   (None, 7, 7

In [24]:
memory = SequentialMemory(limit=1000000, window_length=WINDOW_LENGTH)
processor = AtariProcessor()

In [25]:
policy = LinearAnnealedPolicy(EpsGreedyQPolicy(), attr='eps',
                              value_max=1., value_min=.1, value_test=.05,
                              nb_steps=2000000)

In [26]:
dqn = DQNAgent(model=model, nb_actions=nb_actions, policy=policy,
               memory=memory, processor=processor,
               nb_steps_warmup=50000, gamma=.99,
               target_model_update=10000,
               train_interval=4)
dqn.compile(Adam(learning_rate=.00025), metrics=['mae'])

In [ ]:
# Training part
weights_filename = 'dqn_{}_weights.h5f'.format(env_name)
checkpoint_weights_filename = 'dqn_' + env_name + '_weights_{step}.h5f'
log_filename = 'dqn_{}_log.json'.format(env_name)
callbacks = [ModelIntervalCheckpoint(checkpoint_weights_filename, interval=100000)]
callbacks += [FileLogger(log_filename, interval=100)]

dqn.fit(env, callbacks=callbacks, nb_steps=2000000, log_interval=10000, visualize=False)

dqn.save_weights(weights_filename, overwrite=True)

Training for 2000000 steps ...
Interval 1 (0 steps performed)


/usr/local/lib/python3.11/dist-packages/tf_keras/src/engine/training_v1.py:2354: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


10000/10000 [==============================] - 60s 6ms/step - reward: 0.0066
55 episodes - episode_reward: 1.182 [0.000, 4.000] - ale.lives: 2.920

Interval 2 (10000 steps performed)
10000/10000 [==============================] - 58s 6ms/step - reward: 0.0064
55 episodes - episode_reward: 1.182 [0.000, 4.000] - ale.lives: 2.949

Interval 3 (20000 steps performed)
10000/10000 [==============================] - 58s 6ms/step - reward: 0.0060
56 episodes - episode_reward: 1.036 [0.000, 5.000] - ale.lives: 2.922

Interval 4 (30000 steps performed)
 2808/10000 [=======>......................] - ETA: 42s - reward: 0.0071

/usr/local/lib/python3.11/dist-packages/rl/core.py:206: RuntimeWarning: overflow encountered in scalar add
  self.step += 1


10000/10000 [==============================] - 57s 6ms/step - reward: 0.0064
56 episodes - episode_reward: 1.179 [0.000, 5.000] - ale.lives: 2.981

Interval 5 (40000 steps performed)
10000/10000 [==============================] - 59s 6ms/step - reward: 0.0049
60 episodes - episode_reward: 0.817 [0.000, 4.000] - ale.lives: 2.931

Interval 6 (50000 steps performed)
10000/10000 [==============================] - 59s 6ms/step - reward: 0.0068
55 episodes - episode_reward: 1.236 [0.000, 6.000] - ale.lives: 2.912

Interval 7 (60000 steps performed)
10000/10000 [==============================] - 59s 6ms/step - reward: 0.0064
55 episodes - episode_reward: 1.109 [0.000, 7.000] - ale.lives: 2.985

Interval 8 (70000 steps performed)
10000/10000 [==============================] - 57s 6ms/step - reward: 0.0058
58 episodes - episode_reward: 1.052 [0.000, 4.000] - ale.lives: 2.898

Interval 9 (80000 steps performed)
10000/10000 [==============================] - 58s 6ms/step - reward: 0.0053
59 episo

In [1]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
# Testing part
weights_filename = 'dqn_{}_weights.h5f'.format(env_name)
weights_filename = "dqn_BreakoutDeterministic-v4_weights_2000000.h5f"
from gym import wrappers

env = wrappers.Monitor(env, "/content/drive/", force=True)
env.reset()

weights_filename = "/content/dqn_BreakoutDeterministic-v4_weights_900000.h5f"
dqn.load_weights(weights_filename)
dqn.test(env, nb_episodes=10, visualize=False)

NameError: name 'env_name' is not defined

In [ ]:
import io
import base64
from IPython.display import HTML

video = io.open('/content/drive/openaigym.video.%s.video000008.mp4' % env.file_infix, 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''
    <video width="360" height="auto" alt="test" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>'''
.format(encoded.decode('ascii')))

---